# Observabilidad: la caja negra que sí puedes mirar

**Facsímil 6 · Construir y operar** — capítulo 4 (observabilidad: logs, métricas, trazas y costes).

Un sistema de IA sin observabilidad es una caja negra que solo te avisa de sus problemas... por boca de
los usuarios enfadados. En este cuaderno pones un servicio «en producción» (simulado), registras cada
petición con su latencia, sus tokens y su coste, y construyes el panel que de verdad te dice si el
sistema va bien. Por el camino descubres por qué **mirar la media te engaña** y por qué el percentil 95
es el número que te quita el sueño. Medir latencia, coste y errores por petición es lo que separa
*operar* de *rezar*.

### Qué vas a aprender
- Los tres pilares de la observabilidad: **logs** (qué pasó), **métricas** (cuánto) y **trazas** (por
  dónde).
- A registrar peticiones y resumirlas en métricas que llevan a decisiones.
- Por qué la **media miente** y los **percentiles** (p95, p99) cuentan la verdad.
- A definir **alertas** y **objetivos de servicio** (SLO) con umbrales.

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
import numpy as np
np.random.seed(7)

N = 2000
# La mayoria de peticiones son rapidas, pero una cola larga tarda muchisimo (lo normal).
latencia_ms = np.concatenate([np.random.normal(400, 80, int(N*0.9)),
                              np.random.normal(2200, 600, int(N*0.1))])
np.random.shuffle(latencia_ms); latencia_ms = np.clip(latencia_ms, 50, None)
tokens = np.random.randint(200, 1500, N)
coste = tokens * (0.7/1_000_000)
error = np.random.random(N) < 0.03          # 3% de errores
print(f"{N} peticiones registradas. La traza de las primeras 3:")
for i in range(3):
    print(f"  req#{i}: {latencia_ms[i]:.0f} ms | {tokens[i]} tokens | ${coste[i]:.6f} | {'ERROR' if error[i] else 'ok'}")


2000 peticiones registradas. La traza de las primeras 3:
  req#0: 433 ms | 917 tokens | $0.000642 | ok
  req#1: 3615 ms | 698 tokens | $0.000489 | ok
  req#2: 389 ms | 1378 tokens | $0.000965 | ok


## 1. Los tres pilares: logs, métricas, trazas

Observar no es una sola cosa. Se apoya en tres tipos de datos complementarios:

- **Logs:** el registro de *qué* pasó en cada petición (entrada, salida, error). Útil para investigar un
  caso concreto.
- **Métricas:** números agregados a lo largo del tiempo (*cuánta* latencia, *cuántos* errores, *cuánto*
  coste). Útiles para ver tendencias y disparar alertas.
- **Trazas:** el recorrido de una petición *por dónde* pasó (qué pasos, cuánto tardó cada uno). Útiles
  para ver dónde está el cuello de botella.

Aquí trabajamos sobre todo con métricas, porque son las que deciden si despiertas a alguien a las 3 de
la mañana. Empezamos por las que importan.


In [2]:
print("PANEL DEL SERVICIO")
print(f"  Latencia media:   {latencia_ms.mean():>7.0f} ms")
print(f"  Latencia p50:     {np.percentile(latencia_ms,50):>7.0f} ms  (la mitad va por debajo)")
print(f"  Latencia p95:     {np.percentile(latencia_ms,95):>7.0f} ms  (el 5% peor sufre esto)")
print(f"  Latencia p99:     {np.percentile(latencia_ms,99):>7.0f} ms  (el 1% peor)")
print(f"  Coste total:      ${coste.sum():>7.4f}  ({N} peticiones)")
print(f"  Tasa de error:    {100*error.mean():>7.1f}%")


PANEL DEL SERVICIO
  Latencia media:       574 ms
  Latencia p50:         409 ms  (la mitad va por debajo)
  Latencia p95:        2068 ms  (el 5% peor sufre esto)
  Latencia p99:        2957 ms  (el 1% peor)
  Coste total:      $ 1.1833  (2000 peticiones)
  Tasa de error:        3.0%


## 2. El número que engaña: media frente a percentiles

Mira la diferencia entre la latencia **media** y el **p95**. La media te dice que «todo va razonable»,
mientras un 5% de tus usuarios espera varias veces más. ¿Por qué pasa? Porque la media la arrastran los
muchos casos rápidos y esconde la **cola** de casos lentos. Si solo vigilas la media, no verás el
problema hasta que esos usuarios se vayan. **Operar es vigilar las colas, no los promedios.** Lo
medimos: qué fracción de usuarios sufre por encima de un umbral que la media no delata.


In [3]:
UMBRAL_LENTO = 1500    # ms; por encima, el usuario lo percibe como "lento"
frac_lentos = 100 * (latencia_ms > UMBRAL_LENTO).mean()
print(f"Latencia MEDIA: {latencia_ms.mean():.0f} ms  -> 'parece que todo va bien'")
print(f"Pero el {frac_lentos:.0f}% de las peticiones supera los {UMBRAL_LENTO} ms y se siente lento.")
print(f"La media ({latencia_ms.mean():.0f} ms) ni siquiera llega a ese umbral: te esconde a esos usuarios.")


Latencia MEDIA: 574 ms  -> 'parece que todo va bien'
Pero el 9% de las peticiones supera los 1500 ms y se siente lento.
La media (574 ms) ni siquiera llega a ese umbral: te esconde a esos usuarios.


## 3. Una alerta de verdad: cuándo despertar a alguien

Un panel no sirve si nadie lo mira a las 3 de la mañana. Definimos **objetivos de servicio** (SLO):
umbrales que, si se cruzan, disparan una **alerta**. Por ejemplo: «el p95 no debe pasar de 2 segundos» y
«los errores no deben superar el 5%». Comprobamos si se cumplen.


In [4]:
SLO = {"p95_ms": 2000, "error_pct": 5.0}
medido = {"p95_ms": np.percentile(latencia_ms, 95), "error_pct": 100*error.mean()}
alertas = []
if medido["p95_ms"] > SLO["p95_ms"]:
    alertas.append(f"p95 = {medido['p95_ms']:.0f} ms supera el SLO de {SLO['p95_ms']} ms")
if medido["error_pct"] > SLO["error_pct"]:
    alertas.append(f"errores = {medido['error_pct']:.1f}% supera el SLO de {SLO['error_pct']:.0f}%")
print("Estado frente a los SLO:")
print(f"  p95:     {medido['p95_ms']:.0f} ms  (objetivo <= {SLO['p95_ms']})  -> {'[!] ALERTA' if medido['p95_ms']>SLO['p95_ms'] else 'ok'}")
print(f"  errores: {medido['error_pct']:.1f}%  (objetivo <= {SLO['error_pct']:.0f}%)  -> {'[!] ALERTA' if medido['error_pct']>SLO['error_pct'] else 'ok'}")


Estado frente a los SLO:
  p95:     2068 ms  (objetivo <= 2000)  -> [!] ALERTA
  errores: 3.0%  (objetivo <= 5%)  -> ok


## 4. Experimento: cómo la cola mueve el p95 (y no la media)

Para convencerte de que la media es mala consejera, agrandamos la **cola lenta** (más peticiones
tardísimas) y vemos qué le pasa a la media frente al p95. La media sube de forma moderada; el p95 se
dispara mucho más, porque captura justo a quienes peor lo pasan. Esa es la prueba de por qué se vigilan
percentiles y no solo el promedio.


In [5]:
print("% de peticiones lentas | latencia media | p95")
for frac_cola in [0.05, 0.10, 0.20, 0.30]:
    lat = np.concatenate([np.random.normal(400, 80, int(N*(1-frac_cola))),
                          np.random.normal(2200, 600, int(N*frac_cola))])
    lat = np.clip(lat, 50, None)
    print(f"        {frac_cola*100:>2.0f}%           |   {lat.mean():>6.0f} ms    | {np.percentile(lat,95):>6.0f} ms")
print("\nLa media sube algo, pero el p95 se dispara mucho mas. Por eso un buen panel vigila la cola.")


% de peticiones lentas | latencia media | p95
         5%           |      494 ms    |    653 ms
        10%           |      586 ms    |   2253 ms
        20%           |      764 ms    |   2667 ms
        30%           |      949 ms    |   2814 ms

La media sube algo, pero el p95 se dispara mucho mas. Por eso un buen panel vigila la cola.


## 5. Pruébalo tú

1. **Añade el coste por usuario:** agrupa por un `usuario_id` aleatorio y mira quién te cuesta más. En
   producción, un puñado de usuarios suele concentrar el gasto.
2. **Define tu propio SLO:** «p99 por debajo de 5 s» o «coste de la hora por debajo de X». ¿Se cumpliría
   con estos datos?
3. **Simula un incidente:** sube la tasa de error al 10% en la última parte de las peticiones (un
   despliegue que salió mal) y comprueba que la alerta salta.
4. **Mide el p50, p90, p99 juntos:** la *forma* de la distribución de latencias dice más que cualquier
   número suelto.


## 6. Errores comunes

- **Mirar solo la media.** Esconde a los usuarios que peor lo pasan. Vigila percentiles (p95, p99).
- **No tener alertas.** Un panel que nadie mira no sirve. Define umbrales (SLO) que avisen solos.
- **Medir todo y no decidir nada.** Las métricas existen para tomar decisiones (escalar, optimizar,
  revertir), no para decorar.
- **Olvidar el coste.** En un sistema con LLM, el coste por petición es una métrica de primera, igual
  que la latencia y los errores.


## 7. Qué te llevas

- Observar es **registrar cada petición** (latencia, tokens, coste, error) y resumirla en métricas que
  llevan a decisiones. Tres pilares: logs, métricas, trazas.
- La **media miente**; los **percentiles** (p95, p99) cuentan la verdad de quien peor lo pasa.
- Las **alertas** y los **SLO** convierten un panel en acción: umbrales que avisan antes de que avise el
  usuario.

**Para seguir:** el siguiente cuaderno usa estas métricas para decidir a qué modelo mandar cada petición
(routing) y cuándo tirar de un plan B (fallback) sin pasarse de presupuesto; el capítulo 7 trata los
despliegues progresivos (canary, rollback) que se apoyan en este panel.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*